# **LAB NLP SESSION 4** (Sat, 28 February 2026)
**Sentiment Classification with Naive Bayes**


 Naive Bayes -> merupakan sebuah algoritma untuk klasifikasi yang bekerja berdasarkan peluang (probabilitas). Akan menebak kelas dari suatu data dengan menghitung mana yangpeluang kemunculannya paling tinggi berdasarkan data yang sudah di pelajari (train)


* [] Extract Text Features
* [] Naive Bayes Classifier
* [] Save Model to Pickle

Sentence:

How could she be so blind? After all the times I was there, after every moment I poured my heart into showing her what she meant to me just a friend? That\'s all I am to her?

Well, well, looks like karma finally did its job-can\t say I\m not enjoying the show!

In [28]:
import pandas as pd
import string
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')


from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.probability import FreqDist
from nltk.classify import NaiveBayesClassifier, accuracy

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dianr\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\dianr\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [29]:
# 1. Preprocessing data -> ngebersihim datanya (tokenize, punctuation, stopwords)
# 2. Extracting features -> mengubah teks kalimat menjadi data biner ( False/True) untuk setiap kata yang muncul di kalimat (misal: "saya suka makan nasi" -> {'saya': True, 'suka': True, 'makan': True, 'nasi': True})
# 3. Data Splitting -> 80% training dan 20% testing (uji model)
# 4. Melatih Model Naive Bayes -> menggunakan algoritma Naive Bayes untuk melatih model dengan data training yang sudah diproses
# 5. Deployment -> deploy modelnya (misal: buat API, integrasi ke aplikasi)

In [30]:
# --- Load & Check Data ---

df = pd.read_csv('Emotion_classify_Data.csv')
print(df.head())

                                             Comment Emotion
0  i seriously hate one subject to death but now ...    fear
1                 im so full of life i feel appalled   anger
2  i sit here to write i start to dig out my feel...    fear
3  ive been really angry with r and i feel like a...     joy
4  i feel suspicious if there is no one outside l...    fear


In [31]:

print(df['Emotion'].value_counts())

Emotion
anger    2000
joy      2000
fear     1937
Name: count, dtype: int64


In [32]:

# --- 1. pre-processing data ---

def get_tag(tag):
    if tag == 'jj': # jj adjective
        return 'a'
    elif tag in ['vb', 'nn', 'rb']:
        return tag[0]
    else:
        return 'n'

def lemmatizing(stemmed):
    lemmatizer = WordNetLemmatizer()
    lemmatized = []
    tagged = pos_tag(stemmed)
    
    for word, tag in tagged:
        label = get_tag(tag.lower()) # nerima teks yg udh kita lower
        if label:
            result = lemmatizer.lemmatize(word, label)
            lemmatized.append(result)
            
    return lemmatized

def preprocess_sentence(sentence):
    punctuation = string.punctuation
    eng_stopwords = stopwords.words('english')
    stemmer = SnowballStemmer('english')
    
    word_list = word_tokenize(sentence)
    word_list = [word.lower() for word in word_list]
    removed_punctuation = [token for token in word_list if token not in punctuation]
    remove_stopwords = [token for token in removed_punctuation if token not in eng_stopwords]
    
    stemmed = [stemmer.stem(token) for token in remove_stopwords]
    
    # PERHATIAN: Di foto kamu, fungsi berhenti di sini (return stemmed)
    return stemmed 
    
    # Baris di bawah ini tidak akan tereksekusi karena ada 'return' di atasnya
    lemmatized = lemmatizing(stemmed)
    
    return lemmatized


# KUIS : Rumus Teorema Bayes
# HOKI : STEMMED
# GA HOKI : LEMMATIZING

In [33]:
# 2. Ektracting features -> mengubah teks kalimat menjadi data biner ( False/True) untuk setiap kata yang muncul di kalimat (misal: "saya suka makan nasi" -> {'saya': True, 'suka': True, 'makan': True, 'nasi': True})

x= df['Comment']
y = df['Emotion']

all_comment = ' '.join(x)
all_token = preprocess_sentence(all_comment)

freq_dist = FreqDist(all_token)

print(freq_dist.most_common(10))

[('feel', 6233), ('like', 1013), ('im', 943), ('get', 402), ('time', 360), ('want', 358), ('know', 349), ('go', 347), ('make', 343), ('littl', 326)]


In [34]:
def extract_features(comment): # -> Hasilnya berupa dictionary dengan key adalah kata dan value adalah True/False (apakah kata tersebut ada di comment atau tidak)
# dictionary {word, True/False}
    features = {}
    for word in freq_dist.keys():
        features[word] = (word in comment)
    return features

# comment di sini adalah comment yang sudah di preprocess dari loopingan sebelumnya (stemmed/lemmatized)
feature_sets = [(extract_features(preprocess_sentence(comment)), emotion) for (comment, emotion) in zip(x, y)]


# output feature_sets itu berupa list of tuples, dimana setiap tuple berisi dictionary (hasil dari extract_features) dan label emotionnya [(), (), ...]
# Jadi fungsi dri ekstraksi fitur ini adalah untuk mengubah ulasan yang tadinya berupa kata-kata menjadi daftar 'ada/tidak', lalu dipasangkan dengan label supaya model bisa belajar pola mana yang termasuk ke suatu kelas.

print(feature_sets[0])

# karena naive bayesnya itu hanya menerima tipe data list of tuples
    

({'serious': True, 'hate': True, 'one': True, 'subject': True, 'death': True, 'feel': True, 'reluct': True, 'drop': True, 'im': False, 'full': False, 'life': False, 'appal': False, 'sit': False, 'write': False, 'start': False, 'dig': False, 'think': False, 'afraid': False, 'accept': False, 'possibl': False, 'might': False, 'make': False, 'ive': False, 'realli': False, 'angri': False, 'r': False, 'like': False, 'idiot': False, 'trust': False, 'first': False, 'place': False, 'suspici': False, 'outsid': False, 'raptur': False, 'happen': False, 'someth': False, 'jealous': False, 'becasu': False, 'want': False, 'kind': False, 'love': False, 'true': False, 'connect': False, 'two': False, 'soul': False, 'friend': False, 'mine': False, 'keep': False, 'tell': False, 'morbid': False, 'thing': False, 'dog': False, 'final': False, 'fell': False, 'asleep': False, 'useless': False, 'still': False, 'anxieti': False, 'bit': False, 'annoy': False, 'antsi': False, 'good': False, 'way': False, 'regain': 

In [35]:
# 3. Data Splitting -> 80% training dan 20% testing (uji model)

train_count = int(len(feature_sets) * 0.8)
train_set = feature_sets[:train_count] # ambil index yang pertama 0 ~ train_count yaitu 80% dari data
test_set = feature_sets[train_count:] # ambil index yang setelah train_count sampai akhir data (20% dari data)


In [40]:
# 4. Melatih Model Naive Bayes -> menggunakan algoritma Naive Bayes untuk melatih model dengan data training yang sudah diproses

import random 
random.shuffle(train_set) # untuk menghindari bias data (misal: data yang positif semua di awal, terus negatif semua di akhir) supaya modelnya bisa belajar pola yang lebih general

classifier = NaiveBayesClassifier.train(train_set)

# bisa ngecek seberapa akurasi modelnya dengan test_set
test_accuracy = accuracy(classifier, test_set)
print(f'Akurasi Data Test: {test_accuracy * 100}%')


Akurasi Data Test: 88.46801346801347%


Sentence:

How could she be so blind? After all the times I was there, after every moment I poured my heart into showing her what she meant to me just a friend? That\'s all I am to her?

Well, well, looks like karma finally did its job-can\t say I\m not enjoying the show!

In [39]:
input_text = 'How could she be so blind? After all the times I was there, after every moment I poured my heart into showing her what she meant to me just a friend? That\'s all I am to her?'

preprocessed_text = preprocess_sentence(input_text)
input_features = extract_features(preprocessed_text)
prediction = classifier.classify(input_features)
print(f'Predicted Emotion: {prediction}')
print(prediction) 


Predicted Emotion: anger
anger


SAVE MODEL TO PICKLE
ini buat simpen, ga cuma model, feature set, comment, variabel2 nya juga bisa disimpen

In [43]:
import pickle

# wb -> write binary
with open('naive_bayes_classifier_model.pkl', 'wb') as model:
    pickle.dump(all_token, model)

## **TEST MODEL**

In [44]:
import pickle

# rb -> read binary
with open('naive_bayes_classifier_model.pkl', 'rb') as file:
    all_token = pickle.load(file)

print(all_token[:10])

['serious', 'hate', 'one', 'subject', 'death', 'feel', 'reluct', 'drop', 'im', 'full']
